# Co-occurrence Clustering of Drought Sites

Groups the CONUS streamflow sites into regions based on **temporal co-occurrence of drought events** rather than geographic or hydroclimatic classifications.

## Method overview

1. Expand site-level `(start, end)` drought intervals into a binary daily indicator matrix `X` shaped `(n_sites, n_days)`
2. Compute the **phi coefficient** (= Pearson correlation of binary series) as the similarity metric — accounts for marginal drought frequency differences between sites
3. Convert to a **Euclidean-valid distance**: `D = sqrt(2*(1 - phi))` — achieved by **mean-centring then L2-normalising** each row of `X` before clustering
4. Build a **k-NN geographic connectivity graph** from site lat/lon to constrain merges to spatially proximate sites
5. Run **Ward hierarchical clustering** (via sklearn with connectivity) and inspect the dendrogram to choose `N_CLUSTERS`
6. Map the resulting clusters and compare to McCabe–Wolock regions

**Fixed**: threshold = 30, pct_type = `weibull_jd_mod`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import normalize
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram

# ── Paths ──────────────────────────────────────────────────────────────────────
CONUS_DIR = os.path.dirname(os.path.abspath('cooccurrence_clustering.ipynb'))

SAVEFIG = False

# ── Fixed filters ──────────────────────────────────────────────────────────────
THRESHOLD = 30
PCT_TYPE  = 'weibull_jd_mod'

# ── Tunable parameters ─────────────────────────────────────────────────────────
K_NEIGHBORS  = 5    # k for the geographic connectivity graph
              #       ↑ increase if clustering raises "graph is not fully connected"

# Set after dendrogram inspection (cell 7)
N_CLUSTERS   = 10  # placeholder — fill in after inspecting the dendrogram

# Time window matching the NWM retrospective period
T_START = pd.Timestamp('1979-01-01')
T_END   = pd.Timestamp('2023-12-31')

## 1. Load site metadata (lat/lon)

In [ ]:
# ── Site metadata: lat/lon for all sites ──────────────────────────────────────
site_info = pd.read_parquet(os.path.join(CONUS_DIR, 'site_info.parquet'))
site_info = site_info.set_index('site_no')   # index = zero-padded string, matches obs['site']

# ── McCabe–Wolock: region labels only (for comparison in cells 9 & 10) ─────────
# mw = pd.read_csv(os.path.join(CONUS_DIR, 'McCabe_Wolock_Clusters.csv'),
#                  dtype={'site': str})
# mw = mw.set_index('site')

print(f"site_info sites: {len(site_info):,}")
# print(f"McCabe–Wolock sites: {len(mw):,}")
print(site_info[['latitude', 'longitude']].head())

## 2. Load and filter drought properties

In [ ]:
print("Loading obs drought properties...")
obs = pd.read_csv(os.path.join(CONUS_DIR, 'obs_ref_drought_props_long.csv'), dtype={'site': str})
obs = obs[(obs['threshold'] == THRESHOLD) & (obs['pct_type'] == PCT_TYPE)].copy()
obs['start'] = pd.to_datetime(obs['start'])
obs['end']   = pd.to_datetime(obs['end'])

# zero-padding
obs['site'] = obs['site'].astype(str).str.zfill(8)

# Drop sites not in site_info (e.g. 9-digit IDs absent from metadata)
all_sites = np.array(sorted(obs['site'].unique()))
missing = [s for s in all_sites if s not in site_info.index]
if missing:
    print(f"Dropping {len(missing)} sites not in site_info index: {missing[:5]}{'...' if len(missing) > 5 else ''}")
    obs = obs[~obs['site'].isin(missing)]

sites      = np.array(sorted(obs['site'].unique()))
n_sites    = len(sites)
site_idx   = {s: i for i, s in enumerate(sites)}

print(f"Events: {len(obs):,}  |  Sites: {n_sites:,}")
print(f"Date range: {obs['start'].min().date()} → {obs['end'].max().date()}")

## 3. Build binary daily drought indicator matrix `X`

`X[i, t] = 1` if site `i` is in drought on day `t`, else 0.

In [ ]:
days   = pd.date_range(T_START, T_END, freq='D')
n_days = len(days)
print(f"Time grid: {n_days:,} days ({T_START.date()} – {T_END.date()})")
print(f"Matrix size: {n_sites} × {n_days} = {n_sites * n_days / 1e6:.1f}M cells")
print(f"Memory (float32): {n_sites * n_days * 4 / 1e6:.0f} MB")

X = np.zeros((n_sites, n_days), dtype=np.float32)

for _, row in obs.iterrows():
    i   = site_idx[row['site']]
    s   = max(0, (row['start'] - T_START).days)
    e   = min(n_days - 1, (row['end'] - T_START).days)
    if s <= e:
        X[i, s:e + 1] = 1.0

drought_freq = X.mean(axis=1)
print(f"\nMean drought frequency across sites: {drought_freq.mean():.3f}")
print(f"Min / max: {drought_freq.min():.3f} / {drought_freq.max():.3f}")

## 4. Mean-center then unit-normalise rows

The key relationship we need is:

$$\|\tilde{x}_i - \tilde{x}_j\|_2 = \sqrt{2(1 - \phi_{ij})}$$

where $\phi_{ij}$ is the **phi coefficient** (= Pearson correlation of binary series) and $\tilde{x}$ denotes a **mean-centred, L2-normalised** row.

Simply L2-normalising without centring gives the cosine similarity, **not** Pearson/phi — the two differ because cosine similarity counts only joint drought days (1,1), while phi also accounts for joint non-drought days (0,0) and each site's marginal drought frequency.

Steps:
1. Subtract the row mean: `X_c = X - X.mean(axis=1, keepdims=True)`
2. L2-normalise: `X_norm = X_c / ||X_c||`

Then `X_norm[i] · X_norm[j] = phi_ij` and the Euclidean distance equals `sqrt(2*(1 - phi))`.

In [ ]:
# Drop any sites with zero variance (always/never in drought) before normalizing
nonzero_mask = (X.sum(axis=1) > 0) & (X.sum(axis=1) < n_days)
n_dropped = (~nonzero_mask).sum()
if n_dropped > 0:
    print(f"Dropping {n_dropped} sites with zero variance")
    X       = X[nonzero_mask]
    sites   = sites[nonzero_mask]
    n_sites = len(sites)
    site_idx = {s: i for i, s in enumerate(sites)}

# Step 1: mean-centre each row (removes each site's marginal drought frequency)
X_c = X - X.mean(axis=1, keepdims=True)

# Step 2: L2-normalize each centred row
# Result: X_norm[i] · X_norm[j] = phi_ij  (Pearson correlation of binary indicators)
#         ||X_norm[i] - X_norm[j]|| = sqrt(2*(1 - phi_ij))  ← Ward distance
X_norm = normalize(X_c, norm='l2', axis=1)
print(f"X_norm shape: {X_norm.shape}")

# ── Sanity check ───────────────────────────────────────────────────────────────
phi_01    = float(np.corrcoef(X[0], X[1])[0, 1])
d_euclid  = float(np.linalg.norm(X_norm[0] - X_norm[1]))
d_formula = float(np.sqrt(2 * (1 - phi_01)))
print(f"\nSanity check (sites 0 & 1):")
print(f"  phi coefficient    = {phi_01:.6f}")
print(f"  d_euclid           = {d_euclid:.6f}")
print(f"  sqrt(2*(1-phi))    = {d_formula:.6f}")
print(f"  match: {np.isclose(d_euclid, d_formula)}")

## 5. Build geographic connectivity graph

The k-NN graph (built from lat/lon) constrains Ward linkage to only merge geographically proximate sites. This prevents non-contiguous clusters driven by teleconnections (e.g., Pacific coast grouping with Gulf coast via ENSO).

**Note**: if sklearn warns that the graph is not fully connected, increase `K_NEIGHBORS`.

In [ ]:
# Lat/lon for the working site set (from site_info)
coords = site_info.loc[sites, ['latitude', 'longitude']].values   # (n_sites, 2)

connectivity = kneighbors_graph(
    coords,
    n_neighbors=K_NEIGHBORS,
    include_self=False,
    mode='connectivity'
)
# Symmetrise (kneighbors_graph is directed by default)
connectivity = 0.5 * (connectivity + connectivity.T)

print(f"Connectivity graph: {connectivity.nnz} edges  ({K_NEIGHBORS} neighbors/site)")

In [ ]:
# ── Save edge phi coefficients for connectivity_graph_viz.ipynb ───────────────
# phi_ij = X_norm[i] · X_norm[j]  (Pearson correlation of binary drought series)
# Saved as a 1-D array aligned with the upper-triangle edges of `connectivity`.
from scipy.sparse import coo_matrix as _coo

_c    = _coo(connectivity)
_mask = _c.row < _c.col
_er, _ec = _c.row[_mask], _c.col[_mask]

phi_edges = (X_norm[_er] * X_norm[_ec]).sum(axis=1)   # vectorised dot products

np.save(os.path.join(CONUS_DIR, 'edge_phi_ref.npy'), phi_edges)
print(f"Saved edge_phi_ref.npy  ({len(phi_edges):,} edges)")
print(f"phi range: [{phi_edges.min():.3f}, {phi_edges.max():.3f}]  "
      f"median={np.median(phi_edges):.3f}")


### 5a. Quick `k` checks

In [ ]:
for k_ in range(3, 30):
    g = kneighbors_graph(coords, n_neighbors=k_, mode='connectivity', include_self=False)
    g = 0.5 * (g + g.T)
    n_comp, _ = connected_components(g, directed=False)
    print(f"k={k_:3d}  →  {n_comp} component(s)")
    if n_comp == 1:
        print(f"  ^^^ minimum valid k")
        break

## 6. Fit hierarchical clustering (full tree)

We compute the full tree (`n_clusters=None, distance_threshold=0`) so we can extract the linkage matrix and plot the dendrogram before choosing `N_CLUSTERS`.

In [ ]:
print("Fitting Ward hierarchical clustering (this may take a minute)...")

model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0,    # compute full tree
    linkage='ward',
    connectivity=connectivity,
    compute_distances=True,
)
model.fit(X_norm)

print(f"Done. Leaves: {model.n_leaves_}  |  Connected components: {model.n_connected_components_}")
if model.n_connected_components_ > 1:
    print("WARNING: graph has multiple connected components — increase K_NEIGHBORS")

## 7. Dendrogram inspection

Look for large **jumps in merge distance** (y-axis) — a big gap suggests a natural cut point. The `truncate_mode='lastp'` setting shows only the top `p` merges to keep the plot readable.

In [ ]:
def sklearn_to_linkage_matrix(model):
    """Convert sklearn AgglomerativeClustering output to scipy linkage matrix."""
    n = model.n_leaves_
    children  = model.children_       # (n-1, 2) merge steps
    distances = model.distances_      # (n-1,)   Ward distances at each merge

    # Count observations in each cluster node
    counts = np.zeros(len(children), dtype=int)
    for i, (left, right) in enumerate(children):
        left_count  = 1 if left  < n else counts[left  - n]
        right_count = 1 if right < n else counts[right - n]
        counts[i]   = left_count + right_count

    return np.column_stack([children, distances, counts]).astype(float)


Z = sklearn_to_linkage_matrix(model)

# ── Full merge-distance plot (easier to read than truncated dendrogram) ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: last 50 merge distances as a bar chart
ax = axes[0]
last_n = 60
merge_dists = Z[-last_n:, 2][::-1]   # largest merges first
ax.bar(range(1, last_n + 1), merge_dists, color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Merge step (from top of tree)', fontsize=11)
ax.set_ylabel('Ward linkage distance', fontsize=11)
ax.set_title(f'Top {last_n} merge distances', fontsize=12)
ax.set_xticks(range(1, last_n + 1, 2))
ax.grid(axis='y', alpha=0.4)

# Right: truncated dendrogram
ax = axes[1]
dendrogram(
    Z,
    ax=ax,
    truncate_mode='lastp',
    p=30,
    show_leaf_counts=True,
    leaf_rotation=90,
    color_threshold=0,
)
ax.set_xlabel('Cluster (leaf count in parentheses)', fontsize=10)
ax.set_ylabel('Ward linkage distance', fontsize=11)
ax.set_title('Dendrogram (top 30 merges)', fontsize=12)

plt.tight_layout()
if SAVEFIG:
    plt.savefig(os.path.join(CONUS_DIR, 'cooccurrence_dendrogram.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 20 merge distances (largest gaps suggest natural cut points):")
for k, d in enumerate(Z[::-1, 2][:20], start=1):
    print(f"  {k:>2} clusters → {k+1:>2}: distance = {d:.4f}")

## 7b. Acceleration plot

The **acceleration** at $k$ is the second difference of the merge-distance sequence:

$$a_k = d_{k \to k-1} - d_{(k+1) \to k}$$

where $d_{k \to k-1}$ is the Ward distance of the merge that reduces the tree from $k$ clusters to $k-1$ clusters. A peak in $a_k$ means that going from $k+1$ to $k$ clusters is cheap, but going from $k$ to $k-1$ would be expensive — $k$ is a natural stopping point.

This is the quantitative equivalent of visually hunting for a large gap in the dendrogram.

In [ ]:
MAX_K = 40   # evaluate k = 2 .. MAX_K

# d[k] = Ward distance of the merge that goes from k → k-1 clusters
# In Z (ascending order), that merge is at row -(k-1)
k_vals     = np.arange(2, MAX_K + 1)
merge_cost = np.array([Z[-(k - 1), 2] for k in k_vals])   # monotone decreasing

# acceleration[i] = merge_cost[i] - merge_cost[i+1]
#   = how much more expensive the k→k-1 merge is vs the (k+1)→k merge
# Peak at k* → natural cut at k* clusters
accel   = merge_cost[:-1] - merge_cost[1:]   # length MAX_K - 2
k_accel = k_vals[:-1]                         # k values paired with accel

best_k_accel = int(k_accel[np.argmax(accel)])
print(f"Acceleration peak at k = {best_k_accel}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), tight_layout=True)

ax = axes[0]
ax.plot(k_vals, merge_cost, 'o-', color='steelblue', markersize=4, lw=1.5)
ax.set_xlabel('Number of clusters k', fontsize=11)
ax.set_ylabel('Merge distance (Ward)', fontsize=11)
ax.set_title('Merge distance vs k', fontsize=12)
ax.grid(alpha=0.35)

ax = axes[1]
ax.bar(k_accel, accel, color='tomato', edgecolor='white', linewidth=0.4)
ax.axvline(best_k_accel, color='black', lw=1.5, ls='--', label=f'peak k={best_k_accel}')
ax.set_xlabel('Number of clusters k', fontsize=11)
ax.set_ylabel('Acceleration  $d_{k→k-1} - d_{(k+1)→k}$', fontsize=10)
ax.set_title('Acceleration plot', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.35)

if SAVEFIG:
    plt.savefig(os.path.join(CONUS_DIR, 'cooccurrence_acceleration.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7c. Silhouette score

The **silhouette score** for a site $i$ measures how similar it is to its own cluster vs. the nearest other cluster:

$$s_i = \frac{b_i - a_i}{\max(a_i,\, b_i)}$$

where $a_i$ = mean distance to all sites in the same cluster and $b_i$ = mean distance to all sites in the nearest *other* cluster. $s \in [-1, 1]$; higher is better.

The **mean silhouette** over all sites gives a score for each candidate $k$ — look for a clear peak or a plateau.

Computing the full pairwise distance matrix once and reusing it makes this fast across the $k$ range.

In [ ]:
from sklearn.metrics import silhouette_score

K_RANGE = range(4, 30)   # candidate k values to evaluate

# ── Build full pairwise distance matrix once ───────────────────────────────────
# D[i,j] = sqrt(2*(1 - phi_ij))  for all site pairs
# Using the identity: ||x_norm[i] - x_norm[j]||^2 = 2 - 2*(x_norm[i]·x_norm[j])
print("Computing pairwise distance matrix...")
gram  = X_norm @ X_norm.T                          # (n_sites, n_sites) dot products = phi
D_sq  = np.clip(2.0 - 2.0 * gram, 0, None)        # numerical safety for diagonal
D     = np.sqrt(D_sq).astype(np.float32)
np.fill_diagonal(D, 0.0)
print(f"Distance matrix shape: {D.shape}  |  memory: {D.nbytes / 1e6:.0f} MB")

# ── Silhouette score for each k ────────────────────────────────────────────────
print(f"\nEvaluating silhouette for k = {K_RANGE.start} … {K_RANGE.stop - 1}  (may take ~1 min)...")
sil_scores = {}
for k in K_RANGE:
    m = AgglomerativeClustering(n_clusters=k, linkage='ward', connectivity=connectivity)
    lbl = m.fit_predict(X_norm)
    sil_scores[k] = silhouette_score(D, lbl, metric='precomputed')
    print(f"  k={k:3d}  silhouette={sil_scores[k]:.4f}")

best_k_sil = max(sil_scores, key=sil_scores.get)
print(f"\nBest silhouette at k = {best_k_sil}  ({sil_scores[best_k_sil]:.4f})")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4), tight_layout=True)
ax.plot(list(sil_scores.keys()), list(sil_scores.values()), 'o-', color='steelblue',
        markersize=5, lw=1.5)
ax.axvline(best_k_sil, color='tomato', lw=1.5, ls='--', label=f'best k={best_k_sil}')
ax.set_xlabel('Number of clusters k', fontsize=11)
ax.set_ylabel('Mean silhouette score', fontsize=11)
ax.set_title('Silhouette score vs k  (Ward + geographic connectivity)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.35)

if SAVEFIG:
    plt.savefig(os.path.join(CONUS_DIR, 'cooccurrence_silhouette.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7e. K-sensitivity sweep

Evaluates whether the cluster boundaries are stable across a range of `K_NEIGHBORS` values by accounting for the fact that **different k values may prefer a different natural `N_CLUSTERS`**.

The sweep has three parts:

**Part 1 — Natural N per k** (panel a)  
For each k, fit the full Ward tree and find the acceleration-peak `N_CLUSTERS`. If this shifts a lot with k, then a fixed-N comparison will artificially inflate instability.

**Part 2 — Pairwise NMI at each k's natural partition** (panels b, c)  
Normalized Mutual Information (NMI ∈ [0,1]) quantifies how much two partitions share, regardless of whether they have the same number of clusters. High NMI across k → the underlying geography is stable; the k choice mainly affects granularity.

**Part 3 — Fixed-N ARI** (panel d, only if `N_CLUSTERS` is set)  
ARI comparison at the user-chosen `N_CLUSTERS`. Interpret only if panel a shows that `N_CLUSTERS` is within the natural range for most k values.

**Decision rule:**  
- If consecutive NMI ≥ 0.85 for all steps → k choice does not matter; use `k_min + small buffer`.  
- If NMI drops sharply at a particular k transition → that k changes the regional structure; investigate those two maps before finalising `K_NEIGHBORS`.

Requires: `X_norm`, `coords`, `sklearn_to_linkage_matrix` (cell 7), `adjusted_rand_score` (already imported).

In [ ]:
# from sklearn.metrics import normalized_mutual_info_score

# # ── Sweep configuration ────────────────────────────────────────────────────────
# K_MIN        = 5     # minimum valid k from cell 5a — update if your output differs
# K_STEP       = 3
# K_MAX        = 26
# ACCEL_MAX_K  = 40    # search for natural N up to this many clusters
# ACCEL_MIN_N  = 6     # floor on natural N (avoids trivial k=2 peaks)

# K_SWEEP = sorted(set(range(K_MIN, K_MAX + 1, K_STEP)) | {K_NEIGHBORS})
# print(f"K_SWEEP = {K_SWEEP}\n")

# # ── Part 1: fit full tree at each k → find natural N via acceleration ──────────
# # Requires sklearn_to_linkage_matrix() from cell 7 (dendrogram cell).
# sweep_results = {}   # k → {'natural_N': int, 'labels': array}

# print(f"Fitting full trees for {len(K_SWEEP)} k values...")
# for k in K_SWEEP:
#     g = kneighbors_graph(coords, n_neighbors=k, mode='connectivity', include_self=False)
#     g = 0.5 * (g + g.T)
#     n_comp, _ = connected_components(g, directed=False)
#     if n_comp > 1:
#         print(f"  k={k:3d}  SKIP — graph has {n_comp} components")
#         continue

#     # Full tree
#     m_tree = AgglomerativeClustering(
#         n_clusters=None, distance_threshold=0,
#         linkage='ward', connectivity=g, compute_distances=True,
#     )
#     m_tree.fit(X_norm)
#     Zk = sklearn_to_linkage_matrix(m_tree)

#     # Acceleration peak: d[k->k-1] - d[(k+1)->k], search over ACCEL_MIN_N..ACCEL_MAX_K
#     kv   = np.arange(ACCEL_MIN_N, ACCEL_MAX_K + 1)
#     mc   = np.array([Zk[-(v - 1), 2] for v in kv])
#     acc  = mc[:-1] - mc[1:]
#     nat_N = int(kv[:-1][np.argmax(acc)])

#     # Flat labels at that natural N
#     m_flat = AgglomerativeClustering(n_clusters=nat_N, linkage='ward', connectivity=g)
#     labels_nat = m_flat.fit_predict(X_norm)

#     sweep_results[k] = {'natural_N': nat_N, 'labels': labels_nat}
#     print(f"  k={k:3d}  natural N_CLUSTERS = {nat_N}")

# valid_k  = sorted(sweep_results)
# nat_Ns   = [sweep_results[k]['natural_N'] for k in valid_k]

# # ── Part 2: pairwise NMI between each k's natural partition ───────────────────
# # NMI ∈ [0,1] and does not require identical N, so it's valid here.
# n_k  = len(valid_k)
# NMI  = np.zeros((n_k, n_k))
# for i, ki in enumerate(valid_k):
#     for j, kj in enumerate(valid_k):
#         NMI[i, j] = normalized_mutual_info_score(
#             sweep_results[ki]['labels'], sweep_results[kj]['labels'],
#             average_method='arithmetic',
#         )

# consec_nmi = [NMI[i, i + 1] for i in range(n_k - 1)]

# # ── Part 3: fixed-N ARI comparison (at N_CLUSTERS, if set) ────────────────────
# # Only meaningful if natural_N is roughly stable across k values.
# fixed_labels = {}
# ARI = None
# consec_ari = None
# if N_CLUSTERS is not None:
#     print(f"\nFitting flat Ward at fixed N_CLUSTERS={N_CLUSTERS} for ARI comparison...")
#     for k in valid_k:
#         g = kneighbors_graph(coords, n_neighbors=k, mode='connectivity', include_self=False)
#         g = 0.5 * (g + g.T)
#         m = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward', connectivity=g)
#         fixed_labels[k] = m.fit_predict(X_norm)
#         print(f"  k={k:3d}  done")

#     ARI = np.zeros((n_k, n_k))
#     for i, ki in enumerate(valid_k):
#         for j, kj in enumerate(valid_k):
#             ARI[i, j] = adjusted_rand_score(fixed_labels[ki], fixed_labels[kj])
#     consec_ari = [ARI[i, i + 1] for i in range(n_k - 1)]
# else:
#     print("N_CLUSTERS not set — skipping fixed-N ARI block (run config cell first)")

# # ── Plots ──────────────────────────────────────────────────────────────────────
# n_cols = 4 if ARI is not None else 3
# fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 4), tight_layout=True)

# # (a) Natural N_CLUSTERS vs k
# ax = axes[0]
# ax.plot(valid_k, nat_Ns, 'o-', color='steelblue', markersize=7, lw=2)
# if N_CLUSTERS is not None:
#     ax.axhline(N_CLUSTERS, color='tomato', lw=1.5, ls='--', label=f'current N={N_CLUSTERS}')
# ax.axvline(K_NEIGHBORS, color='gray', lw=1, ls=':', label=f'current k={K_NEIGHBORS}')
# ax.set_xlabel('K_NEIGHBORS', fontsize=10)
# ax.set_ylabel('Natural N_CLUSTERS\n(acceleration peak)', fontsize=10)
# ax.set_title('How optimal N shifts\nwith K_NEIGHBORS', fontsize=11)
# ax.legend(fontsize=8)
# ax.grid(alpha=0.35)

# # (b) Pairwise NMI heatmap (natural partitions)
# ax = axes[1]
# im = ax.imshow(NMI, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
# ax.set_xticks(range(n_k)); ax.set_xticklabels(valid_k, fontsize=8, rotation=45)
# ax.set_yticks(range(n_k)); ax.set_yticklabels(valid_k, fontsize=8)
# ax.set_xlabel('K_NEIGHBORS', fontsize=10); ax.set_ylabel('K_NEIGHBORS', fontsize=10)
# ax.set_title('Pairwise NMI\n(natural partition per k)', fontsize=11)
# plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='NMI')
# for i in range(n_k):
#     for j in range(n_k):
#         ax.text(j, i, f'{NMI[i,j]:.2f}', ha='center', va='center', fontsize=6,
#                 color='black' if NMI[i, j] > 0.4 else 'white')

# # (c) Consecutive NMI line
# ax = axes[2]
# ax.plot(valid_k[:-1], consec_nmi, 'o-', color='steelblue', markersize=6, lw=1.8,
#         label='Consecutive NMI')
# ax.axhline(0.85, color='tomato', lw=1.2, ls='--', label='NMI = 0.85')
# ax.axvline(K_NEIGHBORS, color='gray', lw=1, ls=':', label=f'current k={K_NEIGHBORS}')
# ax.set_ylim(0, 1.05)
# ax.set_xlabel('K_NEIGHBORS  (vs next step)', fontsize=10)
# ax.set_ylabel('NMI', fontsize=10)
# ax.set_title('Consecutive NMI\n(natural partitions)', fontsize=11)
# ax.legend(fontsize=8); ax.grid(alpha=0.35)

# # (d) Fixed-N ARI consecutive (only if N_CLUSTERS is set)
# if ARI is not None:
#     ax = axes[3]
#     ax.plot(valid_k[:-1], consec_ari, 's-', color='darkorange', markersize=6, lw=1.8,
#             label=f'ARI (fixed N={N_CLUSTERS})')
#     ax.axhline(0.85, color='tomato', lw=1.2, ls='--', label='ARI = 0.85')
#     ax.axvline(K_NEIGHBORS, color='gray', lw=1, ls=':', label=f'current k={K_NEIGHBORS}')
#     ax.set_ylim(0, 1.05)
#     ax.set_xlabel('K_NEIGHBORS  (vs next step)', fontsize=10)
#     ax.set_ylabel('ARI', fontsize=10)
#     ax.set_title(f'Consecutive ARI\n(fixed N_CLUSTERS={N_CLUSTERS})', fontsize=11)
#     ax.legend(fontsize=8); ax.grid(alpha=0.35)

# if SAVEFIG:
#     plt.savefig(os.path.join(CONUS_DIR, 'cooccurrence_k_sensitivity.png'), dpi=150, bbox_inches='tight')
# plt.show()

# # ── Printed summary ────────────────────────────────────────────────────────────
# print(f"\n{'k':>5}  {'natural_N':>10}  {'consec_NMI':>12}")
# for i, k in enumerate(valid_k):
#     nmi_str = f'{consec_nmi[i]:.4f}' if i < len(consec_nmi) else '      —'
#     print(f"{k:>5}  {sweep_results[k]['natural_N']:>10}  {nmi_str:>12}")

# n_range = max(nat_Ns) - min(nat_Ns)
# min_nmi = min(consec_nmi) if consec_nmi else float('nan')
# print(f"\nNatural N range: {min(nat_Ns)}–{max(nat_Ns)}  (spread = {n_range})")
# print(f"Min consecutive NMI = {min_nmi:.4f}")
# if n_range <= 3:
#     print("  -> N_CLUSTERS is stable across k. A fixed-N ARI comparison is valid.")
# else:
#     print("  -> N_CLUSTERS shifts with k — different k values prefer different granularities.")
#     print("     Use NMI (panels b/c) as the primary stability metric.")
#     print("     For fixed-N ARI, pick a consensus N near the median of natural_N values.")
#     print(f"     Suggested consensus N: {int(np.median(nat_Ns))}")


## 7d. Domain constraint: minimum regional events per cluster

The statistical methods downstream (Gumbel copula fitting) require a sufficient number of **independent regional drought events** per region. As a rule of thumb, ≥ 30 events is needed for a reliable bivariate fit.

This cell runs `regional_concurrence_intervals` for each candidate $k$ and records the **minimum** number of regional events across all clusters. If any cluster falls below the threshold, $k$ is too large.

This is the most scientifically meaningful constraint — it directly checks whether the chosen $k$ leaves enough data to actually fit the model.

In [ ]:
# import sys
# HYDRO2D_DIR = os.path.join(os.path.dirname(CONUS_DIR), 'hydro2dRP')
# if HYDRO2D_DIR not in sys.path:
#     sys.path.insert(0, HYDRO2D_DIR)
# from regional_declustering import regional_concurrence_intervals

# # ── Parameters (reuse the same values as regional_clustering.ipynb) ────────────
# FRAC_THRESH  = 0.20
# BUFFER_DAYS  = 7
# END_GAP_DAYS = 7
# MIN_EVENTS   = 30   # minimum acceptable regional events per cluster

# K_RANGE_DOMAIN = range(4, 26)

# # Pre-build the obs DataFrame needed by regional_concurrence_intervals
# obs_sub = obs[['site', 'start', 'end', 'severity', 'duration']].copy()

# print(f"Scanning k = {K_RANGE_DOMAIN.start} … {K_RANGE_DOMAIN.stop - 1}  (may take a few minutes)...")

# domain_results = {}   # k → {'min_events': int, 'events_per_cluster': dict}

# for k in K_RANGE_DOMAIN:
#     m   = AgglomerativeClustering(n_clusters=k, linkage='ward', connectivity=connectivity)
#     lbl = m.fit_predict(X_norm)

#     # Map site → cluster label (0-indexed)
#     site_to_cluster = dict(zip(sites, lbl))
#     obs_sub['cluster'] = obs_sub['site'].map(site_to_cluster)

#     events_per_cluster = {}
#     for c, grp in obs_sub.dropna(subset=['cluster']).groupby('cluster'):
#         sub = grp[['site', 'start', 'end']].copy()
#         sub = sub.set_index('site', drop=False)
#         events = regional_concurrence_intervals(
#             sub, frac_thresh=FRAC_THRESH,
#             buffer=BUFFER_DAYS, end_gap=END_GAP_DAYS
#         )
#         events_per_cluster[int(c)] = len(events)

#     min_ev = min(events_per_cluster.values()) if events_per_cluster else 0
#     domain_results[k] = {'min_events': min_ev, 'events_per_cluster': events_per_cluster}
#     flag = '  ✗ BELOW MIN' if min_ev < MIN_EVENTS else ''
#     print(f"  k={k:3d}  min regional events = {min_ev:4d}{flag}")

# # ── Plot ───────────────────────────────────────────────────────────────────────
# min_events_by_k  = [domain_results[k]['min_events']  for k in K_RANGE_DOMAIN]
# mean_events_by_k = [
#     np.mean(list(domain_results[k]['events_per_cluster'].values()))
#     for k in K_RANGE_DOMAIN
# ]

# fig, ax = plt.subplots(figsize=(10, 4), tight_layout=True)
# ax.plot(list(K_RANGE_DOMAIN), mean_events_by_k, 'o-', color='steelblue',
#         markersize=5, lw=1.5, label='Mean events/cluster')
# ax.plot(list(K_RANGE_DOMAIN), min_events_by_k,  's--', color='tomato',
#         markersize=5, lw=1.5, label='Min events/cluster')
# ax.axhline(MIN_EVENTS, color='black', lw=1.2, ls=':', label=f'Minimum threshold ({MIN_EVENTS})')
# ax.set_xlabel('Number of clusters k', fontsize=11)
# ax.set_ylabel('Regional drought events', fontsize=11)
# ax.set_title('Regional events per cluster vs k\n'
#              f'(frac_thresh={FRAC_THRESH}, buffer={BUFFER_DAYS}d, end_gap={END_GAP_DAYS}d)',
#              fontsize=11)
# ax.legend(fontsize=9)
# ax.grid(alpha=0.35)
# if SAVEFIG:
    # plt.savefig(os.path.join(CONUS_DIR, 'cooccurrence_domain_constraint.png'), dpi=150, bbox_inches='tight')
# plt.show()

# # Largest k that still meets the minimum events threshold for all clusters
# valid_k = [k for k in K_RANGE_DOMAIN if domain_results[k]['min_events'] >= MIN_EVENTS]
# print(f"\nValid k values (all clusters ≥ {MIN_EVENTS} events): {valid_k}")
# if valid_k:
#     print(f"Largest valid k: {max(valid_k)}")

## 8. Extract flat cluster labels

Set `N_CLUSTERS` based on the dendrogram above, then run this cell.

In [ ]:
# N_CLUSTERS = 10

In [ ]:
assert N_CLUSTERS is not None, "Set N_CLUSTERS in the config cell after inspecting the dendrogram"

final_model = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    linkage='ward',
    connectivity=connectivity,
    compute_distances=False,
)
labels = final_model.fit_predict(X_norm)

# ── Build site metadata table ──
site_meta = pd.DataFrame({
    'site':       sites,
    'cluster':    labels,
    'lat':        site_info.loc[sites, 'latitude'].values,
    'lon':        site_info.loc[sites, 'longitude'].values,
    # 'mw_region':  mw.reindex(sites)['region'].values,   # NaN for sites not in MW
})
site_meta['cluster'] = site_meta['cluster'] + 1   # 1-indexed

print(f"Sites per cluster:")
print(site_meta.groupby('cluster').size().to_string())
# print(f"\nSites without MW region: {site_meta['mw_region'].isna().sum():,}")

## 9. Map the clusters

In [ ]:
fig, axes = plt.subplots(figsize=(8, 5))

cmap = cm.get_cmap('tab20', N_CLUSTERS)

# Left: co-occurrence clusters
ax = axes
for c in sorted(site_meta['cluster'].unique()):
    sub = site_meta[site_meta['cluster'] == c]
    ax.scatter(sub['lon'], sub['lat'], s=8, color=cmap(c - 1), label=f'C{c} (n={len(sub)})', alpha=0.7)
ax.set_title(f'Co-occurrence clusters (Ward, k={K_NEIGHBORS}, n={N_CLUSTERS}, n_sites={len(site_meta)})', fontsize=11)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(fontsize=6, ncol=2, loc='lower left')

# Right: McCabe–Wolock regions for comparison
# ax = axes[1]
# mw_regions = sorted(site_meta['mw_region'].unique())
# mw_cmap = cm.get_cmap('tab20', len(mw_regions))
# for i, r in enumerate(mw_regions):
#     sub = site_meta[site_meta['mw_region'] == r]
#     ax.scatter(sub['lon'], sub['lat'], s=8, color=mw_cmap(i), label=r, alpha=0.7)
# ax.set_title('McCabe–Wolock regions (reference)', fontsize=11)
# ax.set_xlabel('Longitude')
# ax.set_ylabel('Latitude')
# ax.legend(fontsize=6, ncol=1, loc='lower left')

plt.tight_layout()

if SAVEFIG:
    plt.savefig(os.path.join(CONUS_DIR, 'cooccurrence_map.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Save outputs

In [ ]:
# out_path = os.path.join(CONUS_DIR, f'site_regions_cooccurrence_ref_{N_CLUSTERS}.csv')
# site_meta.to_csv(out_path, index=False)
# print(f"Saved: {out_path}")
# print(site_meta.head())